# DSS-LVR single-experiment playground

This notebook is intentionally **not a batch runner**. Define one experiment in one code cell and run it immediately.

The main function is `run_experiment(...)`. Its arguments control:

- sample size `n` and predictor dimension `p`;
- number of true predictors `n_active`;
- additive vs interaction signal through `interaction`;
- independent vs correlated predictors through `correlated` and `rho`;
- shallow/deep architecture through `hidden_dims`;
- structural mode (`feature_group`, `unit_group`, or `feature_unit_induced_edge`);
- IAF depth/order, warmup, epochs, Monte Carlo sample sizes, learning rate, and gate settings.

Threshold semantics follow the current model implementation: feature groups share one $\tau_F$, unit groups share one $\tau_U$, and `feature_unit_induced_edge` therefore uses two threshold coordinates. No edge/path-specific threshold is introduced.

Nothing is automatically written to disk. Each call prints the training log from `train_grouped_bnn()` and returns only a compact final summary plus feature PIPs.

In [1]:
from pathlib import Path
import math
import sys
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score


def find_project_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "Python" / "simfun.py").exists() and (root / "Python" / "bnn_train.py").exists():
            return root
    raise FileNotFoundError(
        "Run this notebook from inside the NFlow repository containing Python/simfun.py."
    )

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import Python.simfun as sim
import Python.model2 as md
import Python.bnn_train as train

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("project root:", ROOT)
print("device      :", DEVICE)

project root: E:\positron\NFlow
device      : cpu


## Core helpers

The correlated design uses a Gaussian copula so that each predictor still has marginal distribution $U(-\pi,\pi)$. Each active predictor receives correlated inactive competitors. Thus `interaction=True` changes the response surface $f(X)$, while `correlated=True` changes the joint predictor distribution $p(X)$; they are separate experimental factors.

In [2]:
def _torch_generator(seed):
    g = torch.Generator(device=DEVICE)
    g.manual_seed(int(seed))
    return g


def _normal_cdf(z):
    return 0.5 * (1.0 + torch.erf(z / math.sqrt(2.0)))


def correlated_uniform_design(n, p, n_active, seed, rho=0.7, block_size=10):
    """Uniform(-pi, pi) marginals with active-root/null-proxy correlation."""
    n, p, n_active, block_size = map(int, (n, p, n_active, block_size))
    if p < n_active * block_size:
        raise ValueError(
            f"Need p >= n_active * block_size for this correlated design; "
            f"got p={p}, n_active={n_active}, block_size={block_size}."
        )
    if not 0.0 < float(rho) < 1.0:
        raise ValueError("rho must be in (0, 1).")

    g = _torch_generator(seed + 9_100_000 + 131 * p)
    eps = torch.randn(n, p, generator=g, device=DEVICE, dtype=DTYPE)
    z = eps.clone()
    innovation_sd = math.sqrt(1.0 - float(rho) ** 2)

    next_col = n_active
    proxy_pairs = []
    for active_j in range(n_active):
        previous = z[:, active_j]
        for _ in range(block_size - 1):
            z[:, next_col] = float(rho) * previous + innovation_sd * eps[:, next_col]
            proxy_pairs.append((active_j, next_col))
            previous = z[:, next_col]
            next_col += 1

    X = 2.0 * math.pi * _normal_cdf(z) - math.pi
    return X, proxy_pairs


def nonlinear_signal(X, n_active, interaction, seed):
    """Same nonlinear basis as simfun_nonlinear, evaluated on supplied X."""
    choices = ["cos", "sin", "sin2", "cos2"]
    rng = np.random.default_rng(int(seed))
    kinds = choices[:min(int(n_active), 4)]
    if int(n_active) > 4:
        kinds += rng.choice(choices, size=int(n_active) - 4, replace=True).tolist()

    terms = []
    for j, kind in enumerate(kinds):
        x = X[:, j]
        if kind == "cos":
            term = torch.cos(x)
        elif kind == "sin":
            term = torch.sin(x)
        elif kind == "sin2":
            term = torch.sin(x).square()
        else:
            term = torch.cos(x).square()
        terms.append(term)

    signal = torch.stack(terms).sum(dim=0)
    pairs = []
    if bool(interaction):
        candidates = [(j, k) for j in range(int(n_active)) for k in range(j + 1, int(n_active))]
        keep = rng.integers(0, 2, size=len(candidates)).astype(bool)
        if len(candidates) and not keep.any():
            keep[rng.integers(len(candidates))] = True
        pairs = [pair for pair, chosen in zip(candidates, keep) if chosen]
        for j, k in pairs:
            signal = signal + terms[j] * terms[k]
    return signal, kinds, pairs


def make_dataset(
    *, n=600, p=100, n_active=10, interaction=False, correlated=False,
    rho=0.7, corr_block_size=10, target_signal_sd=1.5, sigma2=1.0, seed=400,
):
    """Generate one paired nonlinear simulation entirely in memory."""
    n, p, n_active = int(n), int(p), int(n_active)
    if not 0 < n_active <= p:
        raise ValueError("Require 1 <= n_active <= p.")

    if correlated:
        X, proxy_pairs = correlated_uniform_design(
            n=n, p=p, n_active=n_active, seed=seed, rho=rho, block_size=corr_block_size
        )
    else:
        g = _torch_generator(seed + 8_000_000 + 131 * p)
        X = 2.0 * math.pi * torch.rand(n, p, generator=g, device=DEVICE, dtype=DTYPE) - math.pi
        proxy_pairs = []

    signal, term_types, interaction_pairs = nonlinear_signal(
        X, n_active=n_active, interaction=interaction, seed=seed
    )
    signal = signal - signal.mean()
    sd = signal.std(unbiased=False).clamp_min(1e-8)
    signal = signal * (float(target_signal_sd) / sd)

    noise_g = _torch_generator(seed + 7_000_000 + 131 * p)
    noise = math.sqrt(float(sigma2)) * torch.randn(
        n, generator=noise_g, device=DEVICE, dtype=DTYPE
    )
    y = signal + noise

    # Randomize feature positions while keeping the same support permutation
    # for fixed (seed, p) across interaction/correlation choices.
    rng = np.random.default_rng(int(seed + 5_000_000 + p))
    perm = rng.permutation(p)
    perm_t = torch.as_tensor(perm, device=DEVICE, dtype=torch.long)
    X = X.index_select(1, perm_t)

    truth_source = np.zeros(p, dtype=np.float32)
    truth_source[:n_active] = 1.0
    feature_true = torch.as_tensor(truth_source[perm], device=DEVICE, dtype=DTYPE)
    active_idx = np.flatnonzero(truth_source[perm] > 0.5)

    info = {
        "sim": "nonlinear_custom",
        "seed": int(seed),
        "n": n,
        "p": p,
        "n_active": n_active,
        "active_features": active_idx.tolist(),
        "active_idx": active_idx,
        "feature_true": feature_true.detach().cpu().numpy(),
        "interaction": bool(interaction),
        "interaction_pairs": [(int(j + 1), int(k + 1)) for j, k in interaction_pairs],
        "correlated": bool(correlated),
        "rho": float(rho) if correlated else 0.0,
        "corr_block_size": int(corr_block_size) if correlated else 0,
        "sigma2": float(sigma2),
        "signal_sd": float(signal.std(unbiased=False).item()),
        "term_types": term_types,
        "proxy_pairs": proxy_pairs,
    }
    return X, y, feature_true, signal, info


def split_80_20(X, y, signal, seed=400, train_frac=0.8, diagnostic_frac=0.10):
    n = X.shape[0]
    rng = np.random.default_rng(int(seed + 123_456))
    idx = rng.permutation(n)
    n_train = int(round(float(train_frac) * n))
    train_idx, test_idx = idx[:n_train], idx[n_train:]

    # Diagnostic subset is contained inside training and is used only for printed progress.
    n_diag = max(1, int(round(float(diagnostic_frac) * n_train)))
    diag_idx = train_idx[:n_diag]

    def ix(a):
        return torch.as_tensor(a, device=DEVICE, dtype=torch.long)

    return {
        "X_train": X[ix(train_idx)],
        "y_train": y[ix(train_idx)],
        "X_diag": X[ix(diag_idx)],
        "signal_diag": signal[ix(diag_idx)],
        "X_test": X[ix(test_idx)],
        "signal_test": signal[ix(test_idx)],
    }


def feature_pips(out):
    pip = out["final"].get("feature_pip")
    if pip is None:
        with torch.no_grad():
            pip = out["model"].decoder.feature_semantics(out["final"]["xi"])["active"].float().mean(0)
    if torch.is_tensor(pip):
        pip = pip.detach().cpu().numpy()
    return np.asarray(pip, dtype=float).reshape(-1)


def compact_selection_metrics(pip, feature_true, threshold=0.5):
    truth = np.asarray(feature_true.detach().cpu(), dtype=float).reshape(-1) > 0.5
    active, inactive = truth, ~truth
    selected = pip > float(threshold)
    brier_active = float(np.mean((1.0 - pip[active]) ** 2))
    brier_inactive = float(np.mean(pip[inactive] ** 2)) if inactive.any() else np.nan
    return {
        "tpr": float(np.mean(selected[active])),
        "auroc": float(roc_auc_score(truth.astype(int), pip)) if np.unique(truth).size == 2 else np.nan,
        "brier_bal": float(0.5 * (brier_active + brier_inactive)) if inactive.any() else brier_active,
        "expected_support": float(pip.sum()),
        "selected_support": int(selected.sum()),
    }


## Single experiment function

Edit arguments directly in the experiment cells below. A single call generates its dataset, splits it 80/20, builds the requested DSS-LVR BNN, trains it, and prints a compact final result.

`K_flow=4` means four IAF transformations are stacked and optimized jointly. With `iaf_ordering_scheme="cyclic3"`, their role orderings are $U<V<\tau$, $V<\tau<U$, $\tau<U<V$, $U<V<\tau$. The whole stack receives the same total `epochs`; the orderings are not trained in separate 1200-epoch stages.

In [3]:
def run_experiment(
    *,
    # data
    n=600,
    p=100,
    n_active=10,
    interaction=False,
    correlated=False,
    rho=0.70,
    corr_block_size=10,
    target_signal_sd=1.5,
    sigma2=1.0,
    data_seed=400,
    # network / sparsity
    hidden_dims=(20,),
    selection_mode="feature_group",
    # flow
    K_flow=4,
    flow_type="iaf",
    flow_hidden_units=128,
    flow_hidden_layers=2,
    scale_clip=2.0,
    iaf_ordering_scheme="cyclic3",
    iaf_shuffle_within_role=True,
    gate_type="normalized_requ",
    gate_scale=1.0,
    # optimization
    epochs=1200,
    warmup_epochs=300,
    lr=3e-4,
    R_train=32,
    R_eval=128,
    R_final=2000,
    eval_every=300,
    init_sd=0.5,
    init_loc_jitter=0.05,
    grad_clip=5.0,
    support_threshold=0.5,
    fit_seed=None,
):
    """Run exactly one DSS-LVR simulation fit; no files are written."""
    if fit_seed is None:
        fit_seed = int(data_seed + 100_000 + p + 17 * len(tuple(hidden_dims)))

    X, y, feature_true, signal, info = make_dataset(
        n=n,
        p=p,
        n_active=n_active,
        interaction=interaction,
        correlated=correlated,
        rho=rho,
        corr_block_size=corr_block_size,
        target_signal_sd=target_signal_sd,
        sigma2=sigma2,
        seed=data_seed,
    )
    split = split_80_20(X, y, signal, seed=data_seed)

    print("\nExperiment")
    print("  n / p / active :", n, "/", p, "/", n_active)
    print("  interaction    :", bool(interaction))
    print("  correlated     :", bool(correlated), f"(rho={rho})" if correlated else "")
    print("  hidden_dims    :", tuple(hidden_dims))
    print("  selection_mode :", selection_mode)
    print("  K_flow         :", K_flow, "| ordering:", iaf_ordering_scheme)
    print("  epochs         :", epochs, "| warmup:", warmup_epochs)
    print("  data/fit seed  :", data_seed, "/", fit_seed)

    started = time.perf_counter()
    out = train.train_grouped_bnn(
        X_train=split["X_train"],
        y_train=split["y_train"],
        X_eval=split["X_diag"],
        signal_eval=split["signal_diag"],
        X_final=split["X_test"],
        signal_final=split["signal_test"],
        truth=info,
        reference_decoder=None,
        reference_xi=None,
        selection_mode=selection_mode,
        input_dim=int(p),
        hidden_dims=tuple(hidden_dims),
        out_dim=1,
        family="gaussian",
        sigma2=float(sigma2),
        init_sd=float(init_sd),
        K_flow=int(K_flow),
        flow_type=flow_type,
        flow_hidden_units=int(flow_hidden_units),
        flow_hidden_layers=int(flow_hidden_layers),
        scale_clip=float(scale_clip),
        flow_seed=int(fit_seed + 17),
        iaf_ordering_scheme=iaf_ordering_scheme,
        iaf_shuffle_within_role=bool(iaf_shuffle_within_role),
        gate_type=gate_type,
        gate_scale=float(gate_scale),
        epochs=int(epochs),
        warmup_epochs=int(warmup_epochs),
        lr=float(lr),
        R_train=int(R_train),
        R_eval=int(R_eval),
        R_final=int(R_final),
        eval_every=int(eval_every),
        sampling_timing_repeats=1,
        init_loc_jitter=float(init_loc_jitter),
        grad_clip=float(grad_clip),
        support_threshold=float(support_threshold),
        min_active_draws=25,
        seed=int(fit_seed),
    )
    wall = time.perf_counter() - started

    decoder = out["model"].decoder
    expected_t_dim = {
        "feature_group": 1,
        "unit_group": 1,
        "feature_unit_induced_edge": 2,
    }[selection_mode]
    if int(decoder.t_dim) != expected_t_dim:
        raise RuntimeError(
            f"Unexpected threshold dimension: mode={selection_mode}, "
            f"expected {expected_t_dim}, got {decoder.t_dim}."
        )

    pip = feature_pips(out) if decoder.has_feature_gates else None
    summary = dict(out["final"]["summary"])
    result = {
        "mse": float(summary["mse"]),
        "r2": float(summary["r2"]),
        "wall_time_sec": float(wall),
        "threshold_roles": tuple(decoder.threshold_roles),
        "n_thresholds": int(decoder.t_dim),
    }
    if pip is not None:
        result.update(compact_selection_metrics(pip, feature_true, threshold=support_threshold))
    if "network_density" in summary:
        result["edge_density"] = float(summary["network_density"])
    if "active_path_density" in summary:
        result["path_density"] = float(summary["active_path_density"])
    if "expected_active_units" in summary:
        result["expected_active_units"] = float(summary["expected_active_units"])

    print("\nFinal result")
    print(result)

    if pip is not None:
        pip_table = pd.DataFrame({
            "feature": np.arange(int(p)),
            "truth": feature_true.detach().cpu().numpy().astype(int),
            "pip": pip,
        }).sort_values(["truth", "pip"], ascending=[False, False])
        display(pip_table.head(max(20, int(n_active) + 10)))
    else:
        pip_table = None

    return {
        "result": result,
        "pip": pip,
        "pip_table": pip_table,
        "model": out["model"],
        "xi": out["final"]["xi"],
        "data_info": info,
    }

print("run_experiment ready")

run_experiment ready


## Example 1 — shallow, additive, independent

One code block = one fit. Change any arguments and rerun only this cell.

In [9]:
exp = run_experiment(
    n=500,
    p=1000,
    n_active=10,
    interaction=False,
    correlated=False,
    hidden_dims=(20,),
    selection_mode="feature_group",
    K_flow=4,
    epochs=1200,
    warmup_epochs=300,
    data_seed=400,
)


Experiment
  n / p / active : 500 / 1000 / 10
  interaction    : False
  correlated     : False 
  hidden_dims    : (20,)
  selection_mode : feature_group
  K_flow         : 4 | ordering: cyclic3
  epochs         : 1200 | warmup: 300
  data/fit seed  : 400 / 101417
epoch=0001 phase=repr   valMSE=14.77785 valR2=-4.9498
epoch=0300 phase=repr   valMSE=2.83076 valR2=-0.1397
epoch=0600 phase=select valMSE=1.61515 valR2=0.3497
epoch=0900 phase=select valMSE=1.69371 valR2=0.3181
epoch=1200 phase=select valMSE=1.62903 valR2=0.3441


KeyboardInterrupt: 

## Example 2 — shallow, interaction, independent

In [ ]:
exp = run_experiment(
    n=600,
    p=100,
    n_active=10,
    interaction=True,
    correlated=False,
    hidden_dims=(20,),
    selection_mode="feature_group",
    K_flow=4,
    epochs=1200,
    warmup_epochs=300,
    data_seed=400,
)

## Example 3 — shallow, additive, correlated predictors

In [ ]:
exp = run_experiment(
    n=600,
    p=100,
    n_active=10,
    interaction=False,
    correlated=True,
    rho=0.70,
    corr_block_size=10,
    hidden_dims=(20,),
    selection_mode="feature_group",
    K_flow=4,
    epochs=1200,
    warmup_epochs=300,
    data_seed=400,
)

## Example 4 — deep MLP, feature + unit groups

Use this pattern to change `p`, depth, width, interactions, correlation, or optimization budget independently.

In [ ]:
exp = run_experiment(
    n=600,
    p=100,
    n_active=10,
    interaction=False,
    correlated=False,
    hidden_dims=(20, 20),
    selection_mode="feature_unit_induced_edge",
    K_flow=4,
    epochs=1200,
    warmup_epochs=300,
    data_seed=400,
)

## Blank experiment block

Duplicate this cell whenever you want another manual run.

In [ ]:
exp = run_experiment(
    n=600,
    p=500,
    n_active=10,
    interaction=False,
    correlated=False,
    hidden_dims=(20, 20),
    selection_mode="feature_unit_induced_edge",
    K_flow=4,
    epochs=1200,
    warmup_epochs=300,
    data_seed=400,
)